# DataCleanEnv — RL Training with GRPO

This notebook trains an LLM to clean messy data using GRPO (Group Relative Policy Optimization) from the `trl` library.

## Setup
1. Go to Runtime → Change runtime type → GPU (T4 or better)
2. Run all cells in order
3. Monitor training progress

## Hardware Requirements
- **T4 (16GB)**: Can train Qwen2.5-3B with 4-bit quantization
- **A100 (40GB)**: Can train Qwen2.5-7B or larger
- **L4 (24GB)**: Good balance of speed and memory

## 1. Clone Repository & Install Dependencies

In [ ]:
# ⚠️ IMPORTANT: Replace YOUR_USERNAME with your actual GitHub username!
# Example: !git clone https://github.com/tanmaydesai07/meta-pytorch-hackathon.git

!git clone https://github.com/YOUR_USERNAME/meta-pytorch-hackathon.git
%cd meta-pytorch-hackathon

# Install dependencies
!pip install -q trl transformers accelerate peft bitsandbytes
!pip install -q pandas numpy fastapi fastmcp uvicorn websockets httpx
!pip install -q openenv-core

# Install the data_clean_env package
!pip install -e data_clean_env/

## 2. Check GPU & Configuration

In [ ]:
import torch
import os

# Check GPU
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# Auto-select model based on GPU memory
gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1e9
if gpu_memory >= 40:  # A100
    MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"
elif gpu_memory >= 20:  # L4
    MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"
else:  # T4 (16GB)
    MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"  # Still works with 4-bit!

print(f"Selected model: {MODEL_ID}")

# Training configuration
CONFIG = {
    "learning_rate": 2e-5,
    "num_train_epochs": 2,
    "per_device_train_batch_size": 1,
    "gradient_accumulation_steps": 8,
    "max_prompt_length": 1024,
    "max_completion_length": 2048,
    "num_generations": 4,
    "beta": 0.1,
    "temperature": 0.7,
    "output_dir": "./dataclean-grpo-checkpoint",
    "logging_steps": 5,
    "save_steps": 25,
}

# HuggingFace token (optional - for model push)
# from huggingface_hub import login
# login()  # Uncomment to login interactively

## 3. Generate Diverse Training Data

In [ ]:
# Generate diverse domain-specific datasets for better generalization
!python generate_training_data.py --output-dir data_clean_env/tasks --num-samples 50 --num-datasets 5

# List generated files
import os
tasks_dir = "data_clean_env/tasks"
files = sorted(os.listdir(tasks_dir))
print(f"\n✅ Available task files ({len(files)} total):")
for f in files:
    size = os.path.getsize(os.path.join(tasks_dir, f))
    print(f"   {f} ({size} bytes)")

## 4. Start Environment Server

In [ ]:
import subprocess
import time

# Start the FastAPI server in background
server_process = subprocess.Popen(
    ["uvicorn", "data_clean_env.server.app:app", "--host", "0.0.0.0", "--port", "8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
)

# Wait for server to start
time.sleep(8)
print("✅ Environment server started on http://localhost:8000")

# Test health endpoint
import urllib.request
try:
    response = urllib.request.urlopen("http://localhost:8000/health", timeout=5)
    print(f"✅ Health check passed: {response.status}")
except Exception as e:
    print(f"⚠️ Server starting... will retry: {e}")
    time.sleep(5)
    response = urllib.request.urlopen("http://localhost:8000/health", timeout=5)
    print(f"✅ Health check passed: {response.status}")

## 5. Create Training Dataset

In [ ]:
from datasets import Dataset
import os

# System prompt for data cleaning
SYSTEM_PROMPT = """You are an expert data cleaning agent. Clean messy CSV files by:
1. Reading the file with read_file(path)
2. Using run_python(code) to clean with pandas (remove duplicates, fix missing values, normalize dates, trim whitespace, etc.)
3. Submitting with submit_cleaned_file(path='cleaned_output.csv')

Always save cleaned data with: df.to_csv('cleaned_output.csv', index=False)"""

# Training prompts covering different domains and difficulty levels
prompts = [
    # Easy - customer data
    f"{SYSTEM_PROMPT}\n\nTask: Clean customer dataset. Remove duplicates and handle missing values. File: easy_messy.csv",
    f"{SYSTEM_PROMPT}\n\nTask: Clean this dataset with duplicate rows and missing emails. File: easy_messy.csv",
    
    # Medium - sales records  
    f"{SYSTEM_PROMPT}\n\nTask: Clean sales records. Fix: inconsistent dates, mixed casing, invalid emails, extra whitespace. File: medium_messy.csv",
    f"{SYSTEM_PROMPT}\n\nTask: Normalize dates to YYYY-MM-DD, title case names, validate emails, convert amounts to float. File: medium_messy.csv",
    
    # Hard - product inventory
    f"{SYSTEM_PROMPT}\n\nTask: Clean product inventory. Fix: missing IDs, negative quantities, invalid prices, inconsistent formats. File: hard_messy.csv",
    f"{SYSTEM_PROMPT}\n\nTask: Generate missing IDs, clamp negatives to 0, normalize all data. File: hard_messy.csv",
]

# Add domain-specific prompts if files exist
domain_prompts = [
    # HR domain
    ("hr_messy.csv", "Clean HR employee records. Fix: name casing, salary formats ($, USD), date formats, email issues, missing values."),
    ("hr_messy.csv", "Standardize employee data: title case names, numeric salaries, YYYY-MM-DD dates, valid lowercase emails."),
    
    # Healthcare domain
    ("healthcare_messy.csv", "Clean patient records. Fix: name casing, condition casing, date formats, doctor name formats, status casing."),
    ("healthcare_messy.csv", "Normalize healthcare data: title case all names, standardize dates, fill missing insurance IDs."),
    
    # Finance domain
    ("finance_messy.csv", "Clean financial transactions. Fix: merchant casing, amount formats ($, USD), date formats, category casing."),
    ("finance_messy.csv", "Standardize transaction data: title case merchants, numeric amounts, YYYY-MM-DD dates."),
    
    # Logistics domain
    ("logistics_messy.csv", "Clean shipping records. Fix: origin/destination casing, carrier names, date formats, status casing."),
    ("logistics_messy.csv", "Normalize logistics data: title case locations, standardize dates, fix weight formats."),
    
    # Education domain
    ("education_messy.csv", "Clean student records. Fix: name casing, subject casing, date formats, status casing, missing credits."),
    ("education_messy.csv", "Standardize grade records: title case names/subjects, YYYY-MM-DD dates, fill missing credits."),
]

tasks_dir = "data_clean_env/tasks"
for filename, task_desc in domain_prompts:
    if os.path.exists(os.path.join(tasks_dir, filename)):
        prompts.append(f"{SYSTEM_PROMPT}\n\nTask: {task_desc} File: {filename}")

# Repeat prompts for more training samples
train_dataset = Dataset.from_dict({"prompt": prompts * 8})  # ~128 samples with all domains
print(f"✅ Created training dataset with {len(train_dataset)} samples")
print(f"   Unique prompts: {len(prompts)}")
print(f"   Domains covered: easy, medium, hard + HR, healthcare, finance, logistics, education")

## 6. Define Reward Function (Using Real Environment)

In [ ]:
import re
import json
import asyncio
from data_clean_env.client import DataCleanEnv
from openenv.core.env_server.mcp_types import CallToolAction

ENV_URL = "http://localhost:8000"

def parse_tool_calls(text):
    """Extract tool calls from model output."""
    tool_calls = []
    
    # Pattern 1: Function call format
    patterns = [
        r'(read_file|run_python|write_file|submit_cleaned_file)\s*\(([^)]+)\)',
        r'"name"\s*:\s*"(\w+)".*?"arguments"\s*:\s*({[^}]+})',
    ]
    
    for pattern in patterns:
        for match in re.finditer(pattern, text, re.DOTALL):
            tool_name = match.group(1)
            try:
                args_str = match.group(2)
                # Try to parse as JSON
                if args_str.startswith('{'):
                    args = json.loads(args_str)
                else:
                    # Parse key=value format
                    args = {}
                    for part in args_str.split(','):
                        if '=' in part:
                            k, v = part.split('=', 1)
                            args[k.strip()] = v.strip().strip('"\'')
                        elif ':' in part:
                            k, v = part.split(':', 1)
                            args[k.strip().strip('"')] = v.strip().strip('"\'')
                tool_calls.append((tool_name, args))
            except:
                pass
    
    return tool_calls

async def execute_in_env(completion):
    """Execute completion's tool calls in the environment and get real reward."""
    try:
        env = DataCleanEnv(ENV_URL)
        await env.reset()
        
        tool_calls = parse_tool_calls(completion)
        
        if not tool_calls:
            return 0.0  # No valid tool calls
        
        reward = 0.0
        for tool_name, args in tool_calls[:10]:  # Max 10 steps
            action = CallToolAction(tool_name=tool_name, arguments=args)
            result = await env.step(action)
            if result.done:
                reward = result.reward
                break
        
        return reward
    except Exception as e:
        print(f"Env error: {e}")
        return 0.0

def reward_fn(completions, **kwargs):
    """Reward function using actual environment execution."""
    rewards = []
    loop = asyncio.get_event_loop()
    
    for completion in completions:
        if isinstance(completion, list):
            text = " ".join([t.get("text", str(t)) for t in completion])
        else:
            text = str(completion)
        
        # Try to execute in env for real reward
        try:
            reward = loop.run_until_complete(execute_in_env(text))
        except:
            # Fallback to heuristic if env fails
            reward = 0.0
            if "read_file" in text: reward += 0.1
            if "run_python" in text or "pandas" in text: reward += 0.3
            if "submit_cleaned_file" in text: reward += 0.4
        
        rewards.append(reward)
    
    return rewards

print("✅ Reward function defined (uses real environment when possible)")

## 7. Load Model and Tokenizer

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# 4-bit quantization for memory efficiency
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

# Load model
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
    attn_implementation="flash_attention_2" if torch.cuda.get_device_capability()[0] >= 8 else "eager",
)

print(f"✅ Model loaded: {MODEL_ID}")
print(f"✅ Parameters: {model.num_parameters() / 1e9:.2f}B")
print(f"✅ Memory used: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

## 8. Setup GRPO Trainer

In [ ]:
from trl import GRPOConfig, GRPOTrainer
from peft import LoraConfig

# LoRA for memory-efficient training
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    task_type="CAUSAL_LM",
)

# GRPO training config
training_args = GRPOConfig(
    output_dir=CONFIG["output_dir"],
    learning_rate=CONFIG["learning_rate"],
    num_train_epochs=CONFIG["num_train_epochs"],
    per_device_train_batch_size=CONFIG["per_device_train_batch_size"],
    gradient_accumulation_steps=CONFIG["gradient_accumulation_steps"],
    max_prompt_length=CONFIG["max_prompt_length"],
    max_completion_length=CONFIG["max_completion_length"],
    num_generations=CONFIG["num_generations"],
    beta=CONFIG["beta"],
    temperature=CONFIG["temperature"],
    logging_steps=CONFIG["logging_steps"],
    save_steps=CONFIG["save_steps"],
    save_total_limit=2,
    report_to="none",
    bf16=True,
    optim="paged_adamw_8bit",
    gradient_checkpointing=True,
    warmup_ratio=0.1,
)

# Initialize trainer
trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=reward_fn,
    args=training_args,
    train_dataset=train_dataset,
    peft_config=lora_config,
)

print("✅ GRPO Trainer initialized")
print(f"   Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad) / 1e6:.1f}M")

## 9. Train the Model

In [ ]:
print("🚀 Starting GRPO training...")
print(f"   Model: {MODEL_ID}")
print(f"   Training samples: {len(train_dataset)}")
print(f"   Epochs: {CONFIG['num_train_epochs']}")
print(f"   Effective batch size: {CONFIG['per_device_train_batch_size'] * CONFIG['gradient_accumulation_steps']}")
print(f"   Generations per prompt: {CONFIG['num_generations']}")
print()

# Train!
train_result = trainer.train()

# Save the final model
trainer.save_model(CONFIG["output_dir"])
tokenizer.save_pretrained(CONFIG["output_dir"])

print(f"\n✅ Training complete!")
print(f"✅ Model saved to {CONFIG['output_dir']}")

## 10. Evaluate the Trained Model

In [ ]:
from transformers import pipeline
import torch

# Merge LoRA weights and create inference pipeline
merged_model = trainer.model.merge_and_unload()

pipe = pipeline(
    "text-generation",
    model=merged_model,
    tokenizer=tokenizer,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)

# Test on sample
test_prompt = """You are an expert data cleaning agent. Clean messy CSV files by:
1. Reading the file with read_file(path)
2. Using run_python(code) to clean with pandas
3. Submitting with submit_cleaned_file(path='cleaned_output.csv')

Task: Clean customer dataset with duplicates and missing values. File: easy_messy.csv"""

output = pipe(
    test_prompt,
    max_new_tokens=512,
    temperature=0.7,
    do_sample=True,
    return_full_text=False,
)

print("📝 Generated response:")
print(output[0]["generated_text"])

## 11. Push to HuggingFace Hub (Optional)

In [ ]:
# Uncomment and run to push your trained model to HuggingFace

# from huggingface_hub import login
# login()  # Enter your HF token

# HF_USERNAME = "your-username"  # Change this!
# MODEL_NAME = "dataclean-agent-grpo"

# merged_model.push_to_hub(f"{HF_USERNAME}/{MODEL_NAME}")
# tokenizer.push_to_hub(f"{HF_USERNAME}/{MODEL_NAME}")

# print(f"✅ Model pushed to https://huggingface.co/{HF_USERNAME}/{MODEL_NAME}")

## 12. Cleanup

In [ ]:
# Stop the environment server
server_process.terminate()
server_process.wait()
print("✅ Environment server stopped")